![Johnson & Johnson MedTech Logo](https://raw.githubusercontent.com/Forage-Simulations/Johnson-Johnson-Robotics-Controls/main/github_assets.png)


# Control System Diagnostic Notebook for Robotic Arm

### Objective
This notebook is designed to help identify and resolve response delays in the control code of a robotic arm. You will:

- Diagnose the root cause of delays in command response times.
- Optimize control code for improved performance.
- Document your findings and propose actionable solutions.

### Instructions
Follow the steps outlined in this notebook to diagnose and resolve issues in the robotic arm's control system:

1. **Load Libraries**: Run the provided setup code to load necessary Python libraries.
2. **Run Diagnostic Functions**: Use the `check_response_time` function to measure command response times.
3. **Analyze Findings**: Record observed delays and propose a hypothesis for their cause.
4. **Test Optimizations**: Apply optimization logic and compare results.
5. **Record Results**: Summarize your findings and recommendations in a structured format.

Focus on the `rotate_joint` command, as it has been flagged for delays.


In [1]:
# Import required libraries
import time  # For measuring response times
import numpy as np  # For numerical calculations


In [2]:
# Define a function to measure the response time of commands
def check_response_time(command):
    """Simulates command execution and measures response time."""
    start_time = time.time()
    if command == "rotate_joint":
        time.sleep(0.15)  # Simulate delay for rotate_joint
    elif command == "move_arm":
        time.sleep(0.1)  # Simulate moderate response time
    elif command == "adjust_grip":
        time.sleep(0.05)  # Simulate fast response time
    response_time = time.time() - start_time
    return response_time


In [3]:
# List of commands to test
commands = ["move_arm", "rotate_joint", "adjust_grip"]

# Measure and print response times for each command
print("Testing initial command response times:")
for cmd in commands:
    response_time = check_response_time(cmd)
    print(f"{cmd} response time: {round(response_time, 3)} seconds")


Testing initial command response times:
move_arm response time: 0.102 seconds
rotate_joint response time: 0.15 seconds
adjust_grip response time: 0.051 seconds


### Step 3: Analyze Initial Findings

Record the response times observed for each command. Focus on identifying commands with higher response times.

| Command       | Observed Response Time | Expected Response Time | Notes on Performance   |
|---------------|------------------------|------------------------|-------------------------|
| move_arm      | 0.102 s                | 0.10                   | Slightly above expected (+0.002 s); effectively on target |
| rotate_joint  | 0.15 s                 | 0.18                   | Slowest command, but within its expected limit |
| adjust_grip   | 0.051 s                | 0.09                   | Fastest command; well within its expected limit |

#### Hypothesis:
`rotate_joint` is the slowest command (0.15 s) and is the one flagged for delays, so it is the main candidate for optimization. It is still within its 0.18 s limit, so the delay is not severe. In this notebook the delay comes from a fixed `time.sleep(0.15)`; in a real system, the likely causes would be redundant calculations or blocking waits in the command's code. `move_arm` is only 2 ms over its target, which is most likely small timing overhead.

In [4]:
# Define a function to simulate optimized command execution
def optimized_command(command, improvement_factor=0.2):
    """Simulates optimized command execution."""
    print(f"Optimizing command: {command}")  # Placeholder action
    optimized_response_time = check_response_time(command) * (1 - improvement_factor)
    return optimized_response_time


In [5]:
# Test each command after optimizations
print("\nTesting optimized command response times:")
for cmd in commands:
    optimized_time = optimized_command(cmd)
    print(f"{cmd} optimized response time: {round(optimized_time, 3)} seconds")



Testing optimized command response times:
Optimizing command: move_arm
move_arm optimized response time: 0.08 seconds
Optimizing command: rotate_joint
rotate_joint optimized response time: 0.12 seconds
Optimizing command: adjust_grip
adjust_grip optimized response time: 0.04 seconds


### Step 5: Record Results

Use this section to document your findings from the optimization process.

#### Observations:
- Initial Response Times:
  - move_arm: 0.102 s
  - rotate_joint: 0.15 s
  - adjust_grip: 0.051 s

- Optimized Response Times (about 20% lower; exact values vary by a few milliseconds between runs):
  - move_arm: ~0.082 s
  - rotate_joint: ~0.12 s
  - adjust_grip: ~0.041 s

#### Key Insights:
- Which commands showed improvement after optimization?
  - All three commands improved by about 20%. `rotate_joint` gained the most in absolute terms (about 0.03 s). `move_arm` went from slightly above its expected time (0.102 s vs 0.10 s) to below it (~0.082 s).
- Were any specific optimizations particularly effective?
  - No specific optimization was tested. `optimized_command` only multiplies the measured time by (1 - improvement_factor), so the 20% gain is a simulated assumption, not a measured code change. A real test would need to modify the command's code (for example, remove redundant calculations or blocking waits) and measure again.

### Step 6: Summary and Recommendations

Summarize your findings, including identified issues, optimizations applied, and next steps.

**Identified Issue:** `rotate_joint` is the slowest command (0.15 s) and was flagged for delays. All three commands were already within or very close to their expected response times (`move_arm` was 2 ms over its 0.10 s target).

**Optimization Applied:** A simulated 20% improvement (`improvement_factor=0.2`) was applied to each command. This reduced `rotate_joint` to about 0.12 s, `move_arm` to about 0.082 s and `adjust_grip` to about 0.041 s. Because this is a simulation, the result shows what a 20% gain would look like, not a proven code improvement.

**Next Steps:**
- Replace the simulated delays with the real command code and find the actual cause of the `rotate_joint` delay (redundant calculations, blocking waits).
- Repeat each measurement several times and average the results, since single runs vary slightly. `time.perf_counter()` gives more precise timing than `time.time()`.
- Apply one change at a time and re-measure, so the effect of each change is clear.
- Conduct stress testing to validate performance under high-load scenarios.

### Step 7: Validate the Design Proposal Modifications

The design proposal suggests upgrading the joint actuators for a 15% efficiency improvement. This cell applies that factor to the response times observed in Step 3 and checks the result against the expected response times.

In [6]:
# Validate the proposed efficiency improvement against the expected response times
# Observed times come from Step 3; expected times come from the Step 3 table.
observed = {"move_arm": 0.102, "rotate_joint": 0.15, "adjust_grip": 0.051}
expected = {"move_arm": 0.10, "rotate_joint": 0.18, "adjust_grip": 0.09}
efficiency_factor = 0.15  # proposed efficiency improvement (15%)

names = list(observed)
before = np.array([observed[n] for n in names])
limit = np.array([expected[n] for n in names])
after = before * (1 - efficiency_factor)

print(f"{'Command':<14}{'Before':>8}{'After':>8}{'Expected':>10}{'Margin':>9}  Status")
for n, b, a, l in zip(names, before, after, limit):
    margin = (l - a) / l
    status = "meets standard" if a <= l else "too slow"
    print(f"{n:<14}{b:>8.3f}{a:>8.3f}{l:>10.2f}{margin:>9.1%}  {status}")

print()
print("Commands within expected time before:", int((before <= limit).sum()), "of", len(names))
print("Commands within expected time after: ", int((after <= limit).sum()), "of", len(names))


Command         Before   After  Expected   Margin  Status
move_arm         0.102   0.087      0.10    13.3%  meets standard
rotate_joint     0.150   0.128      0.18    29.2%  meets standard
adjust_grip      0.051   0.043      0.09    51.8%  meets standard

Commands within expected time before: 2 of 3
Commands within expected time after:  3 of 3
